In [15]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from scipy.stats import uniform, randint

In [16]:
# Datasets
datasets = {
    #"heart": ("Data/heart_train.csv", "Data/heart_test.csv"),
    #"diabetes": ("Data/diabetes_train.csv", "Data/diabetes_test.csv"),
    #"cancer": ("Data/diabetes_train.csv", "Data/diabetes_test.csv"),
    "alzheimer": ("Data/alzheimer_train.csv", "Data/alzheimer_test.csv")
}

In [3]:
# Grid of hyperparameters 
param_dist = {
    'max_depth': randint(3, 15),                    # tree depth
    'learning_rate': uniform(0.01, 0.29),           # eta: 0.01-0.3
    'n_estimators': randint(50, 500),               # number of trees
    'subsample': uniform(0.5, 0.5),                 # 0.5-1.0
    'colsample_bytree': uniform(0.5, 0.5),          # 0.5-1.0
    'gamma': uniform(0, 5),                         # min split loss
    'reg_alpha': uniform(0, 1),                     # L1 regularization
    'reg_lambda': uniform(0, 2),                    # L2 regularization
    'min_child_weight': randint(1, 10)              # minimum sum of instance weight
}

In [22]:
all_results = []

for name, (train_path, test_path) in datasets.items():
    print(f"Training: {name}")
    
    # Data load
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]
    
    # XGBoost model
    xgb = XGBClassifier(
        random_state=42,
        eval_metric='auc'  
    )

    
    # Random Search
    random_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_dist,
        n_iter=100,                    
        scoring='roc_auc',
        cv=5,                          
        random_state=42,
        n_jobs=-1
    )
    
    # Fit
    random_search.fit(X_train, y_train)
    
    # Result
    cv_results = pd.DataFrame(random_search.cv_results_)
    
    # Testing on test sets
    for i, params in enumerate(random_search.cv_results_['params']):
        # Training again on parameters from random_search.cv_results_ :(
        model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **params
        )
        model.fit(X_train, y_train)
        
        # Predykcja na zbiorze testowym
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
        
        all_results.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })
    
#Results
results_df = pd.DataFrame(all_results)

Training: alzheimer


In [23]:
#Summary
for dataset in datasets.keys():
    dataset_results = results_df[results_df['dataset'] == dataset]
    best_idx = dataset_results['test_roc_auc'].idxmax()
    best_result = dataset_results.loc[best_idx]
    
    print(f"\n{dataset.upper()}:")
    print(f"  Best test AUC: {best_result['test_roc_auc']:.4f}")
    print(f"  CV AUC: {best_result['cv_roc_auc']:.4f}")
    print(f"  Parameters: {best_result['params']}")


ALZHEIMER:
  Best test AUC: 0.8661
  CV AUC: 0.8742
  Parameters: {'colsample_bytree': 0.8776807051588262, 'gamma': 2.1257793724562237, 'learning_rate': 0.07030308223177476, 'max_depth': 6, 'min_child_weight': 6, 'n_estimators': 240, 'reg_alpha': 0.8422847745949985, 'reg_lambda': 0.8995082667395313, 'subsample': 0.6975751180009072}


In [11]:
results_df.to_csv("Data/xgboost_results.csv", index=False)

In [5]:
results_df= pd.read_csv("xgboost_results.csv")

In [6]:
#results_df.to_csv("wyniki.csv", index=False)
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)


In [7]:
best_per_dataset

,dataset,params,cv_roc_auc,test_roc_auc
0,heart,"{'colsample_bytree': 0.6547638081431639, 'gamm...",0.997322,0.989275
1,mushrooms,"{'colsample_bytree': 0.6872700594236812, 'gamm...",1.000000,1.000000
2,rice,"{'colsample_bytree': 0.5281877483254636, 'gamm...",0.998952,0.999242
3,wine,"{'colsample_bytree': 0.9747603118288211, 'gamm...",0.998468,0.999883


In [8]:
params_df = best_per_dataset["params"].apply(pd.Series)


In [14]:
params_df


,0
0,"{'colsample_bytree': 0.6547638081431639, 'gamm..."
1,"{'colsample_bytree': 0.6872700594236812, 'gamm..."
2,"{'colsample_bytree': 0.5281877483254636, 'gamm..."
3,"{'colsample_bytree': 0.9747603118288211, 'gamm..."


In [15]:
# mean_params = params_df.mean()
# mean_params